# Człowiek w pętli: Bramki przed akcją, kategoryzacja ryzyka i rejestrowanie audytu

README tego rozdziału wprowadza Człowieka w pętli za pomocą krótkiego fragmentu, który pyta użytkownika o `ZAAKCEPTOWANIE` lub `ODRZUCENIE` po tym, jak agent wygenerował odpowiedź. Ten wzorzec jest dobrym punktem wyjścia, ale produkcyjne implementacje HITL zazwyczaj potrzebują trzech dodatkowych elementów:

1. **Bramka przed akcją**, która działa **przed** wykonaniem ryzykownego kroku przez agenta, aby kontrolować koszty, nieodwracalność oraz opóźnienia.
2. **Kategoryzacja ryzyka**, tak aby akcje niskiego ryzyka wykonywały się automatycznie, akcje średniego ryzyka były zatwierdzane partiami, a tylko akcje wysokiego ryzyka wymagały interwencji człowieka.
3. **Dziennik audytu oraz pętla rewizji**, dzięki czemu każda decyzja bramki jest zapisywana w formacie JSONL, a odrzucenie powoduje ponowne wywołanie agenta z ustrukturyzowanym powodem zamiast zwykłego wyświetlania `Rewriting...`.

Ten notatnik buduje każdy z tych elementów na bazie tych samych fundamentów co `06-system-message-framework.ipynb`. Działa end-to-end w trybie `DEMO_MODE = True` (bez potrzeby interaktywnego wprowadzania danych) lub z prawdziwymi wywołaniami `input()` gdy `DEMO_MODE = False`. Uwaga: w trybie DEMO retry trzeciego celu jest zaprogramowany, więc mechanika pętli jest widoczna od początku do końca. Prawdziwa, oparta na rewizji, rekategoryzacja wymaga `DEMO_MODE = False` i operatora.

**Poza zakresem (omawiane w innych lekcjach):** uwierzytelnianie i kontrola dostępu (Lekcja 06, zagrożenie 2), middleware wywołań narzędzi (Lekcja 14, głębokie wprowadzenie do MAF), wzorce debat wieloagentowych.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Wzór 1: Brama przed działaniem

Fragment HITL w README najpierw wywołuje agenta, a następnie prosi użytkownika o zatwierdzenie wyniku. To jest przepływ **po działaniu**. Agent już wykonał działanie, więc koszt wywołania LLM został już poniesiony, a wszelkie skutki uboczne (wysłany e-mail, zapisany wiersz w bazie danych, opublikowany komentarz) już wystąpiły.

Przepływ **przed działaniem** wstawia bramę przed wykonaniem ryzykownego kroku przez agenta. Agent proponuje działanie, brama decyduje, czy je wykonać, i dopiero po zatwierdzeniu skutek uboczny ma miejsce.

| Aspekt | Zatwierdzenie po działaniu (fragment README) | Brama przed działaniem (ten notatnik) |
|---|---|---|
| Kiedy następuje zatwierdzenie? | Po tym, jak agent wygenerował wynik | Przed wykonaniem jakiegokolwiek skutku ubocznego |
| Koszt LLM w przypadku odrzucenia | Już zapłacono | Płaci się tylko za propozycję, nie za działanie |
| Nieodwracalne skutki uboczne | Możliwe (działanie już miało miejsce) | Zapobiegane |
| Jasność audytu | Zatwierdzenie to komunikat wydrukowany | Zatwierdzenie to rekord JSON z timestampem, działaniem, powodem |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Wzorzec 2: Kategoryzacja ryzyka

Nie każda akcja wymaga zatwierdzenia przez człowieka. Tylko do odczytu wyszukiwanie w publicznym API ma inne ryzyko niż wysłanie e-maila do klienta. Traktowanie obu tak samo marnuje uwagę operatora i spowalnia agenta.

Prosty model 3-poziomowy:

| Poziom | Przykłady | Proces zatwierdzania |
|---|---|---|
| `niski` (tylko do odczytu) | Wyszukiwanie w bazie wiedzy, wyszukiwanie opcji lotów, pobieranie publicznej strony internetowej | Automatyczne wykonanie, rejestrowane do audytu |
| `średni` (tania mutacja) | Buforowanie wyniku, inkrementacja licznika, zaplanowanie przypomnienia | Automatyczne wykonanie, ale zbiorcza codzienna weryfikacja |
| `wysoki` (zewnętrzne lub nieodwracalne) | Wysłanie e-maila, obciążenie karty, opublikowanie na publicznym kanale | Blokada do zatwierdzenia przez człowieka |

To jest jeden model kategoryzacji. Systemy produkcyjne często stosują bardziej szczegółowe poziomy (np. poziomy uprawnień AWS IAM, role w dostępie). Poniższa wersja 3-poziomowa to najmniejsza użyteczna wersja dla agenta, który miesza działania tylko do odczytu i te powodujące efekty uboczne.

Poniższy klasyfikator używa heurystyk słów kluczowych, aby demo pozostało deterministyczne i tanie. W systemie produkcyjnym zamieniłbyś to na klasyfikator uczony lub silnik polityk.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Wzorzec 3: Dziennik audytu i pętla rewizji

`print("Response approved.")` nie jest dziennikiem audytu. Dla zaufania każda decyzja bramki powinna być zapisywana jako zdarzenie strukturalne, które można później zapytać, odtworzyć lub dołączyć do przeglądu incydentu.

Dwa elementy:

1. **JSONL tylko do dołączania.** Jeden wiersz na decyzję, z znacznikiem czasu, akcją, poziomem, decyzją, powodem. Łatwe do przeszukania, łatwe do przesłania do prawdziwego magazynu dzienników później.  
2. **Pętla rewizji w przypadku odrzucenia.** Gdy bramka zwraca `deny`, agent ponownie wywołuje siebie z powodem odrzucenia w kontekście, tak aby następna propozycja mogła uniknąć problemu.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatkowe zasoby

Kilka innych publicznych projektów implementuje warianty tych wzorców HITL. Porównaj podejścia, aby znaleźć to, które pasuje do twojego stacku:

- **LangChain** human-in-the-loop narzędzia wrappery ([dokumentacja](https://python.langchain.com/docs/integrations/tools/human_tools)): gotowe wrappery narzędzi, które zatrzymują wykonanie, czekając na dane wejściowe od człowieka.
- **AutoGen** `UserProxyAgent` ([dokumentacja v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); w AutoGen v0.4+ zostało to przeorganizowane): wykorzystuje rolę agenta specjalnie do reprezentowania człowieka w rozmowach wieloagentowych.
- **Semantic Kernel** filtry funkcji ([dokumentacja](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware, które działa wokół każdego wywołania funkcji, odpowiednie do logiki sterującej dostępem.

Każdy projekt obsługuje trzy podwzorce inaczej: LangChain opakowuje je jako narzędzia, AutoGen używa roli agenta, Semantic Kernel stosuje filtry middleware. Przeczytaj jedno lub dwa wdrożenia od początku do końca przed wyborem projektu dla własnego agenta.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
